In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Cleaning up Mesonet Data  
 Because this is meant for an example case the dataset has been both cleaned and subsampled down to 2000 rows. This is the notebook used for cleaning the data and includes some extra code in case one wanted to try a more complex model. 

In [ ]:
# The full data is avaliable on Schooner, path is ~fagg/datasets/
og_csv = "allData1994_2000.csv"
df = pd.read_csv(og_csv)

In [ ]:
# This code could be used for a more sequential model that takes into account the time and station
# df["DATE"] = pd.to_datetime(df[["YEAR", "MONTH", "DAY"]])
# df = df.sort_values(["STID", "DATE"])

In [ ]:
# Check the column names
df.columns

In [ ]:
# These two columns contain the most corrupted data, the sensors were broken
df = df.drop(columns = ['WCMN', 'HTMX'])

In [ ]:
# Check with stations had how many values (they all match which is good)
df['STID'].value_counts()

In [ ]:
# Remove all invalid values
for x in df.columns:
    if(x != "STID"):
       df = df[(df[x] > -995)] 

In [ ]:
# Stations now have mismatching numbers of rows
df['STID'].value_counts()

In [ ]:
# Keep the stations with >1200 rows
counts = df["STID"].value_counts()

keep_stations = counts[counts >= 1200].index

df = df[df["STID"].isin(keep_stations)]

In [ ]:
# Double check
df['STID'].value_counts()

In [ ]:
# Check rain imbalance
plt.hist(df[(df['RAIN'] > 0) & (df['RAIN'] < 0.5)]['RAIN'],bins=20)

In [ ]:
print("No rain days:", (df["RAIN"] == 0).sum())
print("Rain days:", (df["RAIN"] > 0).sum())

In [ ]:
# Edited code from ChatGPT
# For example usage we move down to 2000 rows to make training quick
# Choose number of rain days to keep
rain_sample_size = 1000  # change as needed

# Rain and no-rain subsets
rain_df = df[df['RAIN'] > 0].sample(n=rain_sample_size, random_state=42)
no_rain_df = df[df['RAIN'] == 0].sample(n=rain_sample_size, random_state=42)

# Combine 
balanced_df = pd.concat([rain_df, no_rain_df])

# Quick check
print(balanced_df['RAIN'].gt(0).value_counts())
print("Class counts:")


balanced_df = balanced_df.drop(columns = ['STID'])
balanced_df.to_csv("./data/regression_meso_data.csv", index=False)


# Categorical dataset
balanced_df.loc[balanced_df['RAIN'] == 0, 'RAIN'] = 'NO_RAIN'
balanced_df.loc[balanced_df['RAIN'] != 'NO_RAIN', 'RAIN'] = 'RAINED'
balanced_df.to_csv("./data/categorical_meso_data.csv", index=False)

